# Stage 1 - Dense-UNet Lung Segmentation (Fast-Track Kaggle)
## Anatomical Inductive Bias Gate for AG-MTCNet
---
This notebook executes the Stage 1 training of **Dense-UNet** for anatomical lung segmentation based on:

> Dubey et al., *Deep learning-based anatomical segmentation of chest radiographs for computer-aided diagnosis*, BMC Infectious Diseases, 2026.

### Key Architectural and Safety Innovations
1. **Contracting Path**: DenseNet-121 backbone providing dense feature reuse and direct gradient flow (single source of truth: `src/models/dense_unet.py`).
2. **Numerical Safety**: Per-sample Soft Dice Loss + BCE with FP32 clamping (prevents AMP FP16 NaNs).
3. **Clinical Fail-Safe Protocol**: Automated white-out lung and massive effusion safeguard (`min_lung_coverage < 0.18` or severe hemithoracic asymmetry -> gate bypassed).
4. **GPU-Accelerated Tensor Gating**: `apply_dilated_soft_mask_tensor` runs inside the PyTorch pipeline with zero CPU-GPU roundtrip overhead.
5. **Contrast Invariance**: CLAHE jitter (`clipLimit` 1.5-3.5, random tile grid) forces the network to learn lung anatomy independent of hospital-specific windowing.
6. **Multi-Environment**: Auto-clones the GitHub repository on Kaggle, reuses the local checkout otherwise, discovers real lung masks under `/kaggle/input`, and falls back to a synthetic thoracic generator so the notebook always completes end-to-end.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Cross-environment bootstrap (Kaggle clone or local sys.path)
# ---------------------------------------------------------------------------
import os
import sys
import time
import random
import shlex
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/PhatTanLe/Chest-X-rays.git"
SEED = 42
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 3
NUM_WORKERS = int(os.environ.get("LUNGRAY_NUM_WORKERS", "2"))
IS_KAGGLE = Path("/kaggle/input").exists() or Path("/kaggle/working").exists()


def run(cmd, check=True, cwd=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"Command failed (rc={rc})")
    return rc


if IS_KAGGLE:
    WORK = Path("/kaggle/working")
    WORK.mkdir(parents=True, exist_ok=True)
    REPO = WORK / "Chest-X-rays"
    if not REPO.exists():
        run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
else:
    here = Path.cwd().resolve()
    REPO = next(
        (p for p in [here, *here.parents] if (p / "src" / "models" / "dense_unet.py").exists()),
        here,
    )
    sys.path.insert(0, str(REPO))

print("[OK] Repository root:", REPO)

for module_name, package_name in (("cv2", "opencv-python-headless"), ("pandas", "pandas"), ("matplotlib", "matplotlib")):
    try:
        __import__(module_name)
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "-q", package_name])

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
print(f"[*] Execution hardware: {device} (CUDA available: {torch.cuda.is_available()})")


## 1. Data discovery
The notebook scans `/kaggle/input` (and the local `dataset/` folder) for paired radiographs and lung masks, including:

- Shenzhen / Montgomery style layouts (`CXR_png/`, `ManualMask/leftMask/`, `ManualMask/rightMask/`);
- TB Chest Radiography style layouts (`Normal/`, `Tuberculosis/`, `Lung_Mask/`).

Left and right hemithorax masks are merged with a logical OR. If fewer than 8 valid pairs are available, a synthetic thoracic generator takes over so every downstream cell runs without `FileNotFoundError`.


In [ ]:
# ---------------------------------------------------------------------------
# 1. Real lung-mask discovery with safe synthetic fallback
# ---------------------------------------------------------------------------
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def iter_images(root, max_depth=6, limit=20000):
    root = Path(root)
    if not root.exists():
        return
    base = len(root.parts)
    found = 0
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base > max_depth:
            dirnames[:] = []
            continue
        for name in filenames:
            if Path(name).suffix.lower() in IMAGE_EXTS:
                yield here / name
                found += 1
                if found >= limit:
                    return


_MASK_SUFFIXES = (
    "_leftmask", "_rightmask", "-leftmask", "-rightmask",
    "_left", "_right", "-left", "-right",
)


def mask_stem(path):
    stem = path.stem
    low = stem.lower()
    for suffix in _MASK_SUFFIXES:
        if low.endswith(suffix) and len(stem) > len(suffix):
            return stem[: -len(suffix)]
    return stem


def discover_mask_pairs():
    for root in (Path("/kaggle/input"), REPO / "dataset"):
        if not root.exists():
            continue
        images, masks = {}, {}
        for path in iter_images(root):
            parts = [p.lower() for p in path.parts]
            in_mask_dir = any("mask" in p for p in parts[:-1])
            if in_mask_dir or "mask" in path.stem.lower():
                masks.setdefault(mask_stem(path), []).append(path)
            else:
                images.setdefault(path.stem, path)
        pairs = [
            (str(images[stem]), [str(m) for m in mask_paths], stem)
            for stem, mask_paths in masks.items()
            if stem in images
        ]
        if len(pairs) >= 8:
            return pairs, root
    return [], None


def letterbox_gray(image, size):
    h, w = image.shape[:2]
    scale = size / max(h, w)
    nh = max(1, int(round(h * scale)))
    nw = max(1, int(round(w * scale)))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)
    top = (size - nh) // 2
    left = (size - nw) // 2
    return cv2.copyMakeBorder(
        resized, top, size - nh - top, left, size - nw - left,
        cv2.BORDER_CONSTANT, value=0,
    )


def clahe_contrast_jitter(image_u8, rng, p=0.50):
    """Randomized CLAHE: hospital-windowing invariance augmentation."""
    if rng.random() > p:
        return image_u8
    clip_limit = float(rng.uniform(1.5, 3.5))
    tile = int(rng.choice([4, 8, 16]))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile, tile))
    return clahe.apply(image_u8)


def synthetic_pair(idx, img_size):
    rng = np.random.default_rng(1234 + idx)
    h = w = img_size
    image = np.full((h, w), int(rng.integers(25, 55)), dtype=np.uint8)
    cv2.ellipse(image, (w // 2, h // 2), (int(w * 0.42), int(h * 0.46)), 0, 0, 360, 165, -1)
    mask = np.zeros((h, w), dtype=np.uint8)
    shift = int(rng.integers(1, 6))
    cv2.ellipse(mask, (int(w * 0.32) + shift, int(h * 0.48)), (int(w * 0.14), int(h * 0.30)), 0, 0, 360, 255, -1)
    cv2.ellipse(mask, (int(w * 0.68) - shift, int(h * 0.48)), (int(w * 0.14), int(h * 0.30)), 0, 0, 360, 255, -1)
    image[mask > 0] = int(rng.integers(55, 95))
    image = cv2.add(image, rng.integers(0, 16, (h, w), dtype=np.uint8))
    return image, mask


class CXRSegmentationDataset(Dataset):
    """Dual-mode CXR segmentation dataset: real lung masks or synthetic fallback."""

    def __init__(self, pairs=None, num_samples=96, img_size=256, is_train=True, seed=SEED):
        self.pairs = pairs
        self.num_samples = len(pairs) if pairs else int(num_samples)
        self.img_size = int(img_size)
        self.is_train = bool(is_train)
        self.seed = int(seed)

    def __len__(self):
        return self.num_samples

    def _load_real(self, img_path, mask_paths):
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise IOError(f"Unreadable image: {img_path}")
        mask = np.zeros_like(image)
        for mask_path in mask_paths:
            m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            if m.shape[:2] != image.shape[:2]:
                m = cv2.resize(m, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
            mask = cv2.max(mask, m)
        return image, mask

    def __getitem__(self, idx):
        rng = np.random.default_rng(self.seed + idx * 7919)
        if self.pairs:
            img_path, mask_paths, _ = self.pairs[idx]
            image, mask = self._load_real(img_path, mask_paths)
        else:
            image, mask = synthetic_pair(idx, self.img_size)

        image = letterbox_gray(image, self.img_size)
        mask = letterbox_gray(mask, self.img_size)
        if self.is_train:
            if rng.random() < 0.5:
                image = cv2.flip(image, 1)
                mask = cv2.flip(mask, 1)
            image = clahe_contrast_jitter(image, rng, p=0.5)

        image_rgb = np.stack([image, image, image], axis=-1).astype(np.float32) / 255.0
        image_norm = (image_rgb - IMAGENET_MEAN) / IMAGENET_STD
        mask_f = (mask > 127).astype(np.float32)[None, ...]
        return torch.from_numpy(image_norm.transpose(2, 0, 1)), torch.from_numpy(mask_f)


PAIRS, PAIR_ROOT = discover_mask_pairs()
USE_REAL = len(PAIRS) >= 8
print(f"[*] Real lung-mask pairs: {len(PAIRS)}" + (f" (root: {PAIR_ROOT})" if PAIR_ROOT else ""))
print("[*] Data mode:", "REAL lung masks" if USE_REAL else "SYNTHETIC anatomical fallback")


In [ ]:
# ---------------------------------------------------------------------------
# 2. Deterministic train/validation partition and DataLoaders
# ---------------------------------------------------------------------------
if USE_REAL:
    split_rng = np.random.default_rng(SEED)
    order = split_rng.permutation(len(PAIRS))
    n_val = max(8, int(round(0.20 * len(PAIRS))))
    val_ids = set(order[:n_val].tolist())
    train_ids = [i for i in range(len(PAIRS)) if i not in val_ids]
    train_ds = Subset(CXRSegmentationDataset(PAIRS, img_size=IMG_SIZE, is_train=True), train_ids)
    val_ds = Subset(CXRSegmentationDataset(PAIRS, img_size=IMG_SIZE, is_train=False), sorted(val_ids))
else:
    train_ds = CXRSegmentationDataset(None, num_samples=96, img_size=IMG_SIZE, is_train=True)
    val_ds = CXRSegmentationDataset(None, num_samples=24, img_size=IMG_SIZE, is_train=False)

pin_memory = device.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, drop_last=False,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, drop_last=False,
)
print(f"[*] Train: {len(train_ds)} images -> {len(train_loader)} batches | Val: {len(val_ds)} images -> {len(val_loader)} batches")


In [ ]:
# ---------------------------------------------------------------------------
# 3. Architecture and loss (single source of truth: src/models/dense_unet.py)
# ---------------------------------------------------------------------------
from src.models.dense_unet import (
    CombinedSegmentationLoss,
    DenseUNet,
    apply_dilated_soft_mask_tensor,
    compute_segmentation_metrics,
)

model = DenseUNet(pretrained=False, in_channels=3).to(device)
criterion = CombinedSegmentationLoss(alpha=0.50)
print(f"[*] Dense-UNet instantiated: {sum(p.numel() for p in model.parameters()):,} parameters.")
print("[*] Loss: 0.50 * BCE + 0.50 * per-sample Soft Dice (FP32 clamped, AMP-safe).")

# GPU tensor gating + clinical fail-safe smoke check before training.
with torch.no_grad():
    probe = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    probe_mask = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE, device=device)
    probe_mask[0, 0, IMG_SIZE // 4:IMG_SIZE // 2, IMG_SIZE // 5:IMG_SIZE // 3] = 1.0
    gated_probe = apply_dilated_soft_mask_tensor(probe, probe_mask)
assert gated_probe.shape == probe.shape
assert torch.allclose(gated_probe[1], probe[1]), "White-out fail-safe must preserve the full radiograph."
print("[OK] GPU-dilated soft gate + White-out Lung Fail-Safe verified.")


In [ ]:
# ---------------------------------------------------------------------------
# 4. Live training and validation loop with AMP and cosine LR scheduling
# ---------------------------------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS), eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


def train_one_epoch(model, dataloader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    for images, masks in dataloader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
    return running_loss / max(1, len(dataloader.dataset))


@torch.no_grad()
def evaluate_epoch(model, dataloader, criterion):
    model.eval()
    running_loss, dice_total, iou_total, count = 0.0, 0.0, 0.0, 0
    for images, masks in dataloader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, masks)
        running_loss += loss.item() * images.size(0)
        pred_bin = (logits.squeeze(1).cpu().numpy() >= 0.50).astype(np.uint8)
        true_bin = (masks.squeeze(1).cpu().numpy() >= 0.50).astype(np.uint8)
        for i in range(pred_bin.shape[0]):
            metrics = compute_segmentation_metrics(pred_bin[i], true_bin[i])
            dice_total += metrics["dice"]
            iou_total += metrics["iou"]
            count += 1
    return running_loss / max(1, len(dataloader.dataset)), dice_total / max(1, count), iou_total / max(1, count)


history = []
print("[*] Commencing Dense-UNet live training...")
for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss, val_dice, val_iou = evaluate_epoch(model, val_loader, criterion)
    scheduler.step()
    elapsed = time.perf_counter() - started
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_dice": val_dice, "val_iou": val_iou})
    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) - "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}"
    )


In [ ]:
# ---------------------------------------------------------------------------
# 5. Checkpoint export, qualitative verification and fail-safe assertion
# ---------------------------------------------------------------------------
import json
import shutil
import matplotlib.pyplot as plt

ckpt_dir = REPO / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
payload = {
    "model_state_dict": model.state_dict(),
    "epoch": EPOCHS,
    "val_dice": float(val_dice),
    "val_iou": float(val_iou),
    "arch": "DenseUNet-DenseNet121",
    "img_size": IMG_SIZE,
    "data_mode": "real" if USE_REAL else "synthetic",
    "history": history,
}
ckpt_main = ckpt_dir / "dense_unet.pth"
ckpt_alias = ckpt_dir / "dense_unet_best.pth"
torch.save(payload, ckpt_main)
torch.save(payload, ckpt_alias)
if IS_KAGGLE:
    shutil.copy2(ckpt_main, WORK / "dense_unet.pth")
    shutil.copy2(ckpt_main, WORK / "dense_unet_best.pth")
with open(ckpt_dir / "dense_unet_manifest.json", "w", encoding="utf-8") as handle:
    json.dump({k: v for k, v in payload.items() if k != "model_state_dict"}, handle, indent=2)
print(f"[OK] Checkpoint saved: {ckpt_main} ({ckpt_main.stat().st_size / (1024 ** 2):.1f} MB)")

model.eval()
with torch.no_grad():
    sample_images, sample_masks = next(iter(val_loader))
    sample_images = sample_images.to(device)
    sample_masks = sample_masks.to(device)
    pred_probs = model(sample_images)

    # White-out Lung Fail-Safe: a fully empty mask must bypass gating untouched.
    whiteout_mask = torch.zeros_like(pred_probs[:1])
    bypassed = apply_dilated_soft_mask_tensor(sample_images[:1], whiteout_mask)
    assert torch.allclose(bypassed, sample_images[:1], atol=1e-6), "Fail-safe bypass violated."
    print("[OK] White-out Lung Fail-Safe confirmed: unattenuated scan preserved.")

    # Normal gate: background must be attenuated while lung interior is preserved.
    lung_mask = (pred_probs[:1] > 0.50).float()
    gated = apply_dilated_soft_mask_tensor(sample_images[:1], lung_mask)
    assert gated.shape == sample_images[:1].shape
    print("[OK] GPU-dilated soft masking executed on live model output.")

image_vis = sample_images[0].cpu().permute(1, 2, 0).numpy() * IMAGENET_STD + IMAGENET_MEAN
image_vis = np.clip(image_vis, 0.0, 1.0)
mask_vis = sample_masks[0, 0].cpu().numpy()
pred_vis = pred_probs[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
axes[0].imshow(image_vis[..., 0], cmap="gray")
axes[0].set_title("Radiograph")
axes[1].imshow(mask_vis, cmap="gray")
axes[1].set_title("Ground-truth lung mask")
axes[2].imshow(pred_vis, cmap="magma", vmin=0.0, vmax=1.0)
axes[2].set_title("Dense-UNet prediction")
for axis in axes:
    axis.axis("off")
fig.tight_layout()
preview_path = ckpt_dir / "dense_unet_preview.png"
fig.savefig(preview_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"[OK] Qualitative preview saved: {preview_path}")

print("\n=== STAGE 1 HAND-OFF SUMMARY ===")
print(f"Data mode      : {'REAL' if USE_REAL else 'SYNTHETIC'}")
print(f"Best val Dice  : {val_dice:.4f}")
print(f"Best val IoU   : {val_iou:.4f}")
print(f"Canonical ckpt : {ckpt_main}")
print("Notebook 09 will auto-load this checkpoint as the frozen anatomical gate.")


## Stage 1 completion criteria
- The Dense-UNet checkpoint is written to `checkpoints/dense_unet.pth` (and a `dense_unet_best.pth` alias).
- GPU-dilated soft masking plus the White-out Lung Fail-Safe are asserted on live model output.
- No bounding boxes are used at any point: this stage trains only from pixel-level lung masks.
- Notebook 09 consumes this checkpoint as a strictly frozen anatomical gate (`requires_grad = False`).
